# TP 8 : Les Données Problématiques (Bad Data)
(CORRECTION DÉTAILLÉE)

**Dataset :** Diabetes 130-US Hospitals (UCI ML Repository)  
**Contexte :** 101 766 hospitalisations réelles dans 130 hôpitaux américains (1999-2008).  
**Tâche :** Prédire la réadmission hospitalière à 30 jours.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer

np.random.seed(42)
pd.set_option('display.max_columns', 20)

---
## Chargement des Données Brutes

In [ ]:
print("Chargement du dataset diabetes-130...")
dataset = fetch_ucirepo(id=296)
# encounter_id et patient_nbr sont dans dataset.data.ids (pas dans features)
df_raw = pd.concat([dataset.data.ids, dataset.data.features, dataset.data.targets], axis=1)
print(f"Dimensions : {df_raw.shape}")
df_raw.head(3)

---
## Problème 1 : Valeurs Manquantes Encodées en Texte (`'?'`)

In [ ]:
# Observation du problème
print("Valeurs uniques dans 'race' :")
print(df_raw['race'].value_counts(dropna=False))

print("\nValeurs uniques dans 'weight' :")
print(df_raw['weight'].value_counts(dropna=False).head(10))

In [ ]:
# Compter les '?' par colonne
nb_interrogation = (df_raw == '?').sum()
nb_interrogation = nb_interrogation[nb_interrogation > 0].sort_values(ascending=False)

pct_interrogation = (nb_interrogation / len(df_raw) * 100).round(1)

print("Colonnes contenant '?' :")
print(pd.DataFrame({'Nb ?': nb_interrogation, '% ?': pct_interrogation}))

print("\n💡 Observation :")
print("   'weight' a 97% de '?' → quasi-inutilisable")
print("   'medical_specialty' ~49% et 'payer_code' ~40% → problématiques")
print("   'race', 'diag_1/2/3' → taux plus faibles mais à corriger")

In [ ]:
# Correction : remplacer '?' par NaN
df = df_raw.copy()
df.replace('?', np.nan, inplace=True)

# Vérification
nan_apres = df.isnull().sum()
nan_apres = nan_apres[nan_apres > 0].sort_values(ascending=False)
print("NaN par colonne après conversion :")
print((nan_apres / len(df) * 100).round(1).to_frame('% NaN'))

---
## Problème 2 : Colonnes avec Trop de Valeurs Manquantes

In [ ]:
nan_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
nan_pct_nonzero = nan_pct[nan_pct > 0]

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c' if v > 40 else ('#e67e22' if v > 10 else '#3498db')
          for v in nan_pct_nonzero]
ax.barh(nan_pct_nonzero.index, nan_pct_nonzero.values, color=colors, alpha=0.8)
ax.axvline(x=40, color='red', linestyle='--', linewidth=2, label='Seuil 40% (supprimer)')
ax.axvline(x=10, color='orange', linestyle='--', linewidth=1.5, label='Seuil 10% (imputer avec soin)')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Taux de NaN par colonne (diabetes-130)', fontsize=13)
ax.legend()
for i, (col, val) in enumerate(nan_pct_nonzero.items()):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("\n💡 Décision :")
print("   weight (97%) → supprimer : bruit total")
print("   payer_code (40%) → supprimer ou encoder l'absence")
print("   medical_specialty (49%) → supprimer ou grouper les rares + catégorie 'Inconnu'")

In [ ]:
SEUIL = 40  # %
cols_a_supprimer = nan_pct[nan_pct > SEUIL].index.tolist()
print(f"Colonnes supprimées (>{SEUIL}% NaN) : {cols_a_supprimer}")

df_clean = df.drop(columns=cols_a_supprimer)
print(f"\nDimensions après suppression : {df_clean.shape} (était {df.shape})")

---
## Problème 3 : Déséquilibre des Classes

In [ ]:
df_clean['readmit_30'] = (df_clean['readmitted'] == '<30').astype(int)

counts_3classes = df_clean['readmitted'].value_counts()
counts_binary   = df_clean['readmit_30'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(counts_3classes.index, counts_3classes.values,
            color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.8)
axes[0].set_title('Distribution originale (3 classes)')
axes[0].set_ylabel('Nombre de séjours')
for i, v in enumerate(counts_3classes.values):
    axes[0].text(i, v + 300, f'{v:,}\n({v/len(df_clean):.1%})', ha='center', fontsize=9)

axes[1].bar(['Sain / >30j (0)', 'Réadmis <30j (1)'], counts_binary.values,
            color=['#3498db', '#e74c3c'], alpha=0.8)
axes[1].set_title('Cible binaire : réadmission <30j')
axes[1].set_ylabel('Nombre de séjours')
for i, v in enumerate(counts_binary.sort_index().values):
    axes[1].text(i, v + 300, f'{v:,}\n({v/len(df_clean):.1%})', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Déséquilibre des classes — diabetes-130', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

baseline_acc = counts_binary[0] / len(df_clean)
print(f"\n💡 Accuracy baseline (prédire toujours 0) : {baseline_acc:.2%}")
print("   Un modèle naïf qui ne prédit jamais de réadmission atteint déjà cette accuracy !")

In [ ]:
FEATS_NUM = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

df_work = df_clean[FEATS_NUM + ['readmit_30']].dropna()
X = df_work[FEATS_NUM].values
y = df_work['readmit_30'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ❌ Modèle naïf
model_naif = LogisticRegression(max_iter=500)
model_naif.fit(X_train, y_train)
y_pred_naif = model_naif.predict(X_test)

# ✅ Modèle équilibré
model_bal = LogisticRegression(max_iter=500, class_weight='balanced')
model_bal.fit(X_train, y_train)
y_pred_bal = model_bal.predict(X_test)

print("=== MODÈLE NAÏF ===")
print(classification_report(y_test, y_pred_naif, target_names=['Non réadmis', 'Réadmis <30j']))

print("=== MODÈLE ÉQUILIBRÉ (class_weight='balanced') ===")
print(classification_report(y_test, y_pred_bal, target_names=['Non réadmis', 'Réadmis <30j']))

# Matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_naif,
    display_labels=['Non réadmis', 'Réadmis <30j'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Modèle NAÏF\n(Accuracy haute, Recall classe 1 ≈ 0)', fontsize=10)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_bal,
    display_labels=['Non réadmis', 'Réadmis <30j'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title("Modèle ÉQUILIBRÉ (class_weight='balanced')\nDétecte vraiment les réadmissions", fontsize=10)

plt.suptitle('Impact du déséquilibre des classes', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Conclusion :")
print(f"   Naïf    — Accuracy: {accuracy_score(y_test, y_pred_naif):.2%}, "
      f"F1 (réadmis): {f1_score(y_test, y_pred_naif):.3f}")
print(f"   Équilibré — Accuracy: {accuracy_score(y_test, y_pred_bal):.2%}, "
      f"F1 (réadmis): {f1_score(y_test, y_pred_bal):.3f}")
print("   Le modèle naïf ne détecte presque aucun patient réadmis !")
print("   En médecine, un Faux Négatif (patient à risque non détecté) est très grave.")

---
## Problème 4 : Patients en Doublon (Multiple Encounters)

In [ ]:
print(f"Séjours totaux    : {len(df_clean):,}")
print(f"Patients uniques  : {df_clean['patient_nbr'].nunique():,}")
print(f"Séjours 'en trop' : {len(df_clean) - df_clean['patient_nbr'].nunique():,}")

sejours_par_patient = df_clean.groupby('patient_nbr').size()
print("\nDistribution du nombre de séjours par patient :")
vc = sejours_par_patient.value_counts().sort_index()
print(vc.head(10).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(vc.index[:12], vc.values[:12], color='steelblue', alpha=0.8)
ax.set_xlabel('Nombre de séjours par patient')
ax.set_ylabel('Nombre de patients')
ax.set_title('Patients multi-séjours dans diabetes-130')
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

In [ ]:
# Dédoublonnage : garder le premier séjour de chaque patient
df_dedup = (
    df_clean
    .sort_values('encounter_id')
    .drop_duplicates(subset='patient_nbr', keep='first')
)

print(f"Avant dédoublonnage : {len(df_clean):,} lignes")
print(f"Après dédoublonnage : {len(df_dedup):,} lignes")
print(f"Lignes supprimées   : {len(df_clean) - len(df_dedup):,}")

print(f"\nTaux réadmission AVANT : {df_clean['readmit_30'].mean():.2%}")
print(f"Taux réadmission APRÈS : {df_dedup['readmit_30'].mean():.2%}")

print("\n💡 Explication :")
print("   Les patients réadmis (par définition ceux qui reviennent) ont mécaniquement")
print("   plusieurs séjours → leur sur-représentation avant dédoublonnage gonfle le taux.")
print("   Après dédoublonnage, le taux reflète mieux la prévalence réelle.")

---
## Problème 5 : Variables à Haute Cardinalité

In [ ]:
cols_cat = ['race', 'gender', 'age', 'admission_type_id',
            'discharge_disposition_id', 'diag_1', 'diag_2', 'diag_3']

print("Cardinalité des variables catégorielles :")
for col in cols_cat:
    if col in df_dedup.columns:
        n_unique = df_dedup[col].nunique(dropna=True)
        top3 = df_dedup[col].value_counts().head(3).index.tolist()
        print(f"  {col:35s} : {n_unique:5d} valeurs  | top 3 : {top3}")

In [ ]:
def grouper_rares(serie, top_n=10, label_autre='Autre'):
    """Conserve les top_n modalités les plus fréquentes, regroupe le reste en 'Autre'."""
    top_categories = serie.value_counts().nlargest(top_n).index
    return serie.where(serie.isin(top_categories), other=label_autre)


# Application sur diag_1
print(f"diag_1 avant regroupement : {df_dedup['diag_1'].nunique(dropna=True)} modalités")

diag1_reduit = grouper_rares(df_dedup['diag_1'], top_n=20)
print(f"diag_1 après regroupement (top 20) : {diag1_reduit.nunique(dropna=True)} modalités")
print("\nDistribution après regroupement :")
print(diag1_reduit.value_counts().head(10))

# Visualisation de la distribution des codes avant/après
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top15_brut = df_dedup['diag_1'].value_counts().head(15)
axes[0].barh(top15_brut.index, top15_brut.values, color='steelblue', alpha=0.8)
axes[0].set_title(f'diag_1 BRUT (top 15 / {df_dedup["diag_1"].nunique()} catégories)')

top15_reduit = diag1_reduit.value_counts().head(15)
colors = ['#e74c3c' if v == 'Autre' else '#2ecc71' for v in top15_reduit.index]
axes[1].barh(top15_reduit.index, top15_reduit.values, color=colors, alpha=0.8)
axes[1].set_title(f'diag_1 REGROUPÉ (top 15 / {diag1_reduit.nunique()} catégories)')

plt.suptitle('Réduction de la cardinalité de diag_1', fontsize=12)
plt.tight_layout()
plt.show()

---
## Exercice Final : Pipeline de Nettoyage Complet

In [ ]:
def grouper_rares(serie, top_n=10, label_autre='Autre'):
    top_categories = serie.value_counts().nlargest(top_n).index
    return serie.where(serie.isin(top_categories), other=label_autre)


def nettoyer_diabetes(df_input, verbose=True):
    """
    Pipeline de nettoyage complet pour diabetes-130.
    Retourne X (array numpy), y (array numpy).
    """
    df = df_input.copy()

    # Étape 1 : '?' → NaN
    df.replace('?', np.nan, inplace=True)
    if verbose: print(f"[1] Après '?' → NaN : {len(df)} lignes")

    # Étape 2 : Cible binaire
    df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

    # Étape 3 : Supprimer colonnes trop creuses (> 40% NaN)
    nan_pct = df.isnull().mean()
    cols_drop = nan_pct[nan_pct > 0.40].index.tolist()
    # Garder les colonnes identifiants pour l'étape 4
    cols_drop_clean = [c for c in cols_drop if c not in ['patient_nbr', 'encounter_id']]
    df.drop(columns=cols_drop_clean, inplace=True)
    if verbose: print(f"[3] Colonnes supprimées ({'>40% NaN'}) : {cols_drop_clean}")

    # Étape 4 : Dédoublonnage (premier séjour par patient)
    df.sort_values('encounter_id', inplace=True)
    df.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)
    if verbose: print(f"[4] Après dédoublonnage : {len(df)} lignes")

    # Supprimer identifiants (pas de valeur prédictive)
    df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted'], inplace=True)

    # Étape 5 : Réduire la cardinalité des codes diagnostics
    for col in ['diag_1', 'diag_2', 'diag_3']:
        if col in df.columns:
            df[col] = grouper_rares(df[col], top_n=20)
    if verbose: print(f"[5] diag_1/2/3 réduits à 21 modalités max")

    # Étape 6 : Encoder les variables catégorielles
    cols_cat = df.select_dtypes(include='object').columns.tolist()
    cols_cat = [c for c in cols_cat if c != 'readmit_30']
    for col in cols_cat:
        df[col] = df[col].fillna('Inconnu')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    if verbose: print(f"[6] {len(cols_cat)} variables catégorielles encodées")

    # Étape 7 : Imputer les NaN restants (features numériques)
    y = df.pop('readmit_30').values
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(df)
    if verbose: print(f"[7] NaN restants imputés | X final : {X.shape}")

    return X, y

In [ ]:
print("=== Application du pipeline de nettoyage ===")
X_clean, y_clean = nettoyer_diabetes(df_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

rf_clean = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_clean.fit(X_tr, y_tr)
y_pred_clean = rf_clean.predict(X_te)

# Comparaison avec baseline (features numériques brutes)
FEATS_NUM = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]
df_base = df_raw.replace('?', np.nan)[FEATS_NUM + ['readmitted']].dropna()
df_base['y'] = (df_base['readmitted'] == '<30').astype(int)
X_base = df_base[FEATS_NUM].values
y_base = df_base['y'].values

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)
rf_base = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_base.fit(X_tr_b, y_tr_b)
y_pred_base = rf_base.predict(X_te_b)

# Résultats
metriques = pd.DataFrame({
    'Données brutes\n(8 features num.)': {
        'Accuracy': accuracy_score(y_te_b, y_pred_base),
        'F1 (réadmis)': f1_score(y_te_b, y_pred_base),
        'Recall (réadmis)': classification_report(y_te_b, y_pred_base, output_dict=True)['1']['recall'],
        'N features': X_base.shape[1],
        'N lignes train': len(X_tr_b),
    },
    'Données nettoyées\n(pipeline complet)': {
        'Accuracy': accuracy_score(y_te, y_pred_clean),
        'F1 (réadmis)': f1_score(y_te, y_pred_clean),
        'Recall (réadmis)': classification_report(y_te, y_pred_clean, output_dict=True)['1']['recall'],
        'N features': X_clean.shape[1],
        'N lignes train': len(X_tr),
    }
}).T

print("\n📊 Comparaison données brutes vs nettoyées :")
print(metriques.round(4))

# Graphique comparatif
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_plot = ['Accuracy', 'F1 (réadmis)', 'Recall (réadmis)']
for i, metric in enumerate(metrics_plot):
    vals = [metriques.loc['Données brutes\n(8 features num.)', metric],
            metriques.loc['Données nettoyées\n(pipeline complet)', metric]]
    bars = axes[i].bar(['Brutes', 'Nettoyées'], vals,
                       color=['#e74c3c', '#2ecc71'], alpha=0.8, edgecolor='black')
    axes[i].set_ylim(max(0, min(vals) - 0.05), min(1.0, max(vals) + 0.07))
    axes[i].set_title(metric, fontsize=12)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Impact du nettoyage des données — diabetes-130', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Conclusion :")
print("   Le pipeline complet exploite plus de features (catégorielles + numériques)")
print("   et applique toutes les étapes de nettoyage → meilleures performances,")
print("   surtout sur le Recall (détection des vrais cas de réadmission).")

---
## Réponses aux Questions de Réflexion

**1. Pourquoi encoder les manquants comme `'?'` est dangereux ?**  
`'?'` est interprété comme une **catégorie valide** par sklearn (LabelEncoder, OneHotEncoder). Le modèle apprend des patterns liés à cette "catégorie" qui n'existe pas vraiment, et peut généraliser à tort. De plus, les méthodes d'imputation de sklearn (SimpleImputer, KNNImputer) ne reconnaissent pas `'?'` comme manquant.

**2. Quand garder une colonne à 97% de NaN ?**  
Si la présence/absence d'une valeur est elle-même informative : créer une feature binaire `weight_connu` (1 si renseigné, 0 sinon). Dans ce dataset, le fait que le poids ne soit pas mesuré peut indiquer un protocole ou un contexte particulier. On peut donc garder `weight_connu` même si `weight` est supprimé.

**3. Pourquoi préférer le Recall à l'Accuracy en médecine ?**  
Un Faux Négatif (patient réadmis prédit "sain") est bien plus grave qu'un Faux Positif (patient sain prédit "à risque"). Le Recall de la classe positive mesure directement la capacité à détecter les cas réels. L'Accuracy est dominée par la classe majoritaire (~89%) et masque l'échec sur la classe minoritaire.

**4. Risques du regroupement des codes ICD-9 et alternatives ?**  
- **Risque :** on perd l'information de diagnostic précis. Des codes très différents (ex: fracture vs infarctus) peuvent se retrouver dans `'Autre'`.
- **Alternative 1 :** Regrouper par **chapitre ICD-9** (ex: 390-459 = maladies circulatoires) → réduction sémantique plutôt que statistique.
- **Alternative 2 :** Target encoding (voir TP7) → remplace chaque code par son taux moyen de réadmission.
- **Alternative 3 :** Embedding de codes médicaux (Word2Vec sur séquences de codes) → approche avancée.